# Protenix V1 RNA Validation Inference

This notebook is a fresh inference notebook for your local Protenix checkpoints.
It does not modify the existing notebooks. The flow mirrors the smaller validation-style
pattern of the Ribonanza reference notebook, but uses your official Protenix v1 setup,
your local checkpoint files, and the Kaggle RNA validation cache.


In [1]:
pwd

'/scratch/work/sethih1/RNA-prediction/sin-Stanford-RNA-3D-Folding-2/kaggle-new-solution/notebooks'

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

code_dir_candidates = [
    (Path.cwd().parent / "code").resolve(),
    (Path.cwd() / "kaggle-new-solution" / "code").resolve(),
]
CODE_DIR = next(path for path in code_dir_candidates if path.exists())
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import protenix_v1_rna_inference_helpers as helpers

helpers.configure_notebook_logging()


In [3]:
PROTENIX_REPO = None
RNA_DATA_DIR = None
CACHE_ROOT = None
CHECKPOINT_PATH = None
MODEL_NAME = "protenix_base_default_v1.0.0"

USE_MSA = True
USE_RNA_MSA = True
N_EXAMPLES = 4
TARGET_IDS = None

DIFFUSION_SAMPLES = 5
DIFFUSION_STEPS = 20
MC_DROPOUT_APPLY_RATE = 0.0

OUTPUT_DIR = (Path("../notebooks_output/protenix_v1_rna_inference")).resolve()


In [4]:
checkpoint_df = helpers.discover_checkpoints(PROTENIX_REPO)
selected_checkpoint = helpers.choose_checkpoint(CHECKPOINT_PATH, checkpoint_df)
data_dir, validation_sequences, validation_labels = helpers.load_validation_tables(RNA_DATA_DIR)

if TARGET_IDS is None:
    selected_target_ids = validation_sequences["target_id"].astype(str).tolist()[:N_EXAMPLES]
else:
    selected_target_ids = list(TARGET_IDS)

print(f"Data dir: {data_dir}")
print(f"Selected checkpoint: {selected_checkpoint}")
print(f"Targets: {selected_target_ids}")

display(checkpoint_df.head(20))
display(validation_sequences[validation_sequences["target_id"].isin(selected_target_ids)].copy())


Data dir: /scratch/phys/sin/rna-dataset
Selected checkpoint: /scratch/phys/sin/rna-dataset/models/1199_ema_0.995.pt
Targets: ['8ZNQ', '9IWF', '9JGM', '9MME']


,source,path,filename,parent,is_ema,step,size_mb,mtime,source_rank
0,official_train_dir,/scratch/phys/sin/rna-dataset/models/1199_ema_...,1199_ema_0.995.pt,models,True,1199,4222.36,2026-03-25 00:38:22,0
1,official_train_dir,/scratch/phys/sin/rna-dataset/models/1099_ema_...,1099_ema_0.995.pt,models,True,1099,4222.36,2026-03-24 22:33:26,0
2,official_train_dir,/scratch/phys/sin/rna-dataset/models/999_ema_0...,999_ema_0.995.pt,models,True,999,4222.35,2026-03-24 20:27:06,0
3,official_train_dir,/scratch/phys/sin/rna-dataset/models/899_ema_0...,899_ema_0.995.pt,models,True,899,4222.35,2026-03-24 18:20:48,0
4,official_train_dir,/scratch/phys/sin/rna-dataset/models/799_ema_0...,799_ema_0.995.pt,models,True,799,4222.35,2026-03-24 16:18:22,0
5,official_train_dir,/scratch/phys/sin/rna-dataset/models/699_ema_0...,699_ema_0.995.pt,models,True,699,4222.35,2026-03-24 14:11:01,0
6,official_train_dir,/scratch/phys/sin/rna-dataset/models/599_ema_0...,599_ema_0.995.pt,models,True,599,4222.35,2026-03-24 12:03:34,0
7,official_train_dir,/scratch/phys/sin/rna-dataset/models/499_ema_0...,499_ema_0.995.pt,models,True,499,4222.35,2026-03-24 09:56:19,0
8,official_train_dir,/scratch/phys/sin/rna-dataset/models/399_ema_0...,399_ema_0.995.pt,models,True,399,4222.35,2026-03-24 07:51:57,0
9,official_train_dir,/scratch/phys/sin/rna-dataset/models/299_ema_0...,299_ema_0.995.pt,models,True,299,4222.35,2026-03-24 05:45:46,0


,target_id,sequence,temporal_cutoff,description,stoichiometry,all_sequences,ligand_ids,ligand_SMILES
0,8ZNQ,ACCGUGACGGGCCUUUUGGCUAUACGCGGU,2025-06-04,Solution structure of the complex of naphthyri...,A:1,>8ZNQ_1|Chain A[auth A]|RNA (30-MER)|\nACCGUGA...,NAZ,Cc1ccc2ccc(nc2n1)NC(=O)CCNCCC(=O)NCc3ccc4c(n3)...
1,9IWF,GGUGUAUAAGCUCAUUAAUACGGUUUGAGCGUUUCGACCAGGCAAC...,2025-06-04,crystal structure of P. beijingensis xanthine-...,A:1,>9IWF_1|Chain A[auth A]|P. beijingensis xanthi...,GTP;MG;XAN,c1nc2c(n1[C@H]3[C@@H]([C@@H]([C@H](O3)CO[P@](=...
2,9JGM,GGAAGGGGAGUAACUUCAUUGCCGGUCGAUCGUCAUUACGAUGUGU...,2025-06-04,The Escherichia coli yybp riboswitch as a tand...,C:2,">9JGM_1|Chains A[auth C], C[auth D]|yybP ribos...",MG;MN,[Mg+2];[Mn+2]
3,9MME,UAUUUGAAUCAUACCUGCGAUCAACUCGAUGAAUAAAGUACGCCAG...,2025-06-04,ROOLfirm-octamer-wild type,U:8,">9MME_1|Chains A[auth U], B[auth Y], C[auth c]...",K;MG,[K+];[Mg+2]


In [5]:
runner, runner_configs = helpers.load_runner(
    checkpoint_path=selected_checkpoint,
    model_name=MODEL_NAME,
    protenix_repo=PROTENIX_REPO,
    dtype="bf16",
    use_msa=USE_MSA,
    use_rna_msa=USE_RNA_MSA,
    n_diffusion_samples=DIFFUSION_SAMPLES,
    n_diffusion_steps=DIFFUSION_STEPS,
    mc_dropout_apply_rate=MC_DROPOUT_APPLY_RATE,
    output_dir=OUTPUT_DIR,
)

bundles = helpers.prepare_validation_examples(
    selected_target_ids,
    cache_root=CACHE_ROOT,
    protenix_repo=PROTENIX_REPO,
    use_msa=USE_MSA,
    use_rna_msa=USE_RNA_MSA,
)

bundle_overview = pd.DataFrame(
    [
        {
            "target_id": bundle["target_id"],
            "length": len(bundle["sequence"]),
            "n_conformers": int(bundle["residue_coordinate_multi"].shape[0]),
            "used_msa": bundle["used_msa"],
            "was_cropped": bundle["was_cropped"],
        }
        for bundle in bundles
    ]
)
display(bundle_overview)


2026-03-25 00:46:03,512 [/scratch/phys/sin/rna-dataset/venv/px-kaggle/lib/python3.11/site-packages/rdkit/__init__.py:22] INFO rdkit: Enabling RDKit 2025.09.3 jupyter extensions
2026-03-25 00:46:17,430 [/scratch/work/sethih1/RNA-prediction/sin-Stanford-RNA-3D-Folding-2/external/Protenix-v1-official/runner/inference.py:246] INFO runner.inference: Distributed environment: world size: 1, global rank: 0, local rank: 0
2026-03-25 00:46:17,431 [/scratch/work/sethih1/RNA-prediction/sin-Stanford-RNA-3D-Folding-2/external/Protenix-v1-official/runner/inference.py:98] INFO root: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-a996a8cd-e068-5040-a7ff-83a92288d733]
2026-03-25 00:46:17,432 [/scratch/work/sethih1/RNA-prediction/sin-Stanford-RNA-3D-Folding-2/external/Protenix-v1-official/runner/inference.py:127] INFO root: Finished environment initialization.


train scheduler 16.0
inference scheduler 16.0
Diffusion Module has 16.0


FileNotFoundError: [Errno 2] No such file or directory: '/home/sethih1/common/components.cif'

In [ ]:
results = helpers.run_validation_inference(runner, bundles)
summary_df = helpers.summarize_results(results)
summary_display = summary_df.drop(columns=["aligned_prediction"])

display(summary_display)

best_per_target = (
    summary_display
    .sort_values(["target_id", "aligned_rmsd", "ranking_score"], ascending=[True, True, False])
    .groupby("target_id", as_index=False)
    .first()
)
display(best_per_target)


In [ ]:
TARGET_TO_PLOT = selected_target_ids[0]
PREDICTION_ROW = 0

result = helpers.get_result(results, TARGET_TO_PLOT)
display(result["metrics"].drop(columns=["aligned_prediction"]))

row = result["metrics"].iloc[PREDICTION_ROW]
pred_idx = int(row["prediction_index"])
gt_idx = int(row["best_gt_index"])
aligned_pred = row["aligned_prediction"]
true_coords = result["bundle"]["residue_coordinate_multi"][gt_idx]

helpers.plot_prediction_vs_truth(
    aligned_pred,
    true_coords,
    title=f"{TARGET_TO_PLOT}: prediction {pred_idx} vs validation conformer {gt_idx}",
)


In [ ]:
helpers.plot_rna_3d_interactive(
    result["bundle"]["residue_coordinate_multi"][gt_idx],
    title=f"{TARGET_TO_PLOT}: validation conformer {gt_idx}",
)


In [ ]:
predictions_dir = helpers.save_prediction_artifacts(result, OUTPUT_DIR)
print(f"Saved prediction files to: {predictions_dir}")
